# Traveling Salesman Problem via Row Generation (Subtour Elimination)

## Overview

We solve the TSP using **row generation**: we iteratively solve a relaxed MIP and add **subtour elimination constraints (SECs)** until the solution forms a single cycle.

We model the TSP over **undirected edges** $\{i,j\}$ in the Euclidean plane. 

### Master Problem

At each iteration we solve:

$$
\begin{aligned}
\min \quad & \sum_{\{i,j\} \in E} d_{ij}\, x_{ij} \\
\text{s.t.} \quad & \sum_{j \neq i} x_{\{i,j\}} = 2 & \forall\, i \in V \quad & (\text{two edges touch each node}) \\
& \sum_{\{i,j\} \subseteq S} x_{ij} \leq |S| - 1 & \forall\, S \subsetneq V,\; 2 \leq |S| \leq |V|-1 \quad & (\text{SECs, added lazily}) \\
& x_{ij} \in \{0, 1\} & \forall\, \{i,j\} \in E
\end{aligned}
$$

where $E = \{\{i,j\} : i < j\}$ is the set of all undirected edges.

### Row Generation Loop

| Step | Action |
|------|---------|
| **Solve** | Solve the master MIP with all constraints added so far |
| **Detect** | Find all (undirected) subtours — connected components — in the optimal solution |
| **Feasible?** | If there is exactly one tour visiting all nodes → **STOP** |
| **Cut** | For every subtour $S$ detected, add SEC $\sum_{\{i,j\}\subseteq S} x_{ij} \leq |S|-1$ |
| **Repeat** | Go back to Solve |

Because each SEC rules out at least one infeasible subtour and the number of possible subtours is finite, the algorithm terminates in a finite number of iterations.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pyomo highspy numpy matplotlib networkx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import networkx as nx
import time
import warnings
warnings.filterwarnings('ignore')

import pyomo.environ as pyo

try:
    import highspy
except ImportError:
    raise ImportError("Install highspy:  pip install highspy")

try:
    solver_test = pyo.SolverFactory('appsi_highs')
    print(f"Pyomo appsi_highs solver available: {solver_test.available()}")
except Exception as e:
    raise ImportError(f"Pyomo appsi_highs not available: {e}")

## 1. Problem Instance

In [ ]:
# ── Generate a small-to-medium TSP instance ───────────────────────────────────
rng = np.random.default_rng(seed=7)
N = 100                                         # number of cities
coords = rng.uniform(0, 100, size=(N, 2))        # (x, y) coordinates

# Euclidean distance matrix (symmetric: D[i,j] == D[j,i])
def euclidean_distance_matrix(coords: np.ndarray) -> np.ndarray:
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt((diff ** 2).sum(-1))

D = euclidean_distance_matrix(coords)            # D[i,j] = distance between i and j
np.fill_diagonal(D, np.inf)                      # no self-loops

# Edge set — undirected, each pair counted once (i < j)
EDGES = [(i, j) for i in range(N) for j in range(N) if i < j]

# ── Visualize cities ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=50, zorder=3, color='steelblue',
           edgecolors='white', linewidths=1.2)
for i, (x, y) in enumerate(coords):
    ax.annotate(str(i), (x, y), textcoords='offset points', xytext=(6, 4),
                fontsize=10, fontweight='bold')
ax.set_title(f"{N}-city TSP instance", fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Cities: {N}   |   Undirected edges: {len(EDGES)}")

## 2. Master Problem

We model the master MIP in Pyomo with **undirected binary edge variables** $x_{ij}$, $i < j$. We keep a **growing list of SEC cuts** `sec_cuts` — each entry is a `frozenset` of node indices forming a subtour, and the corresponding constraint is:

$$\sum_{\substack{\{i,j\} \in E \\ i \in S,\, j \in S}} x_{ij} \leq |S| - 1$$

A single degree constraint per node replaces the former pair of in/out-degree constraints:

$$\sum_{j \neq i} x_{\{i,j\}} = 2 \qquad \forall\, i \in V$$

In [ ]:
def build_master_problem(
    N: int,
    D: np.ndarray,
    EDGES: list,
    sec_cuts: list[frozenset],
) -> pyo.ConcreteModel:
    """
    Build the undirected TSP master MIP with the current set of SEC cuts.

    Parameters
    ----------
    N        : number of cities
    D        : N×N symmetric distance matrix
    EDGES    : list of (i,j) pairs with i < j  (undirected edges)
    sec_cuts : list of frozensets, each defining an SEC

    Returns
    -------
    model : Pyomo ConcreteModel ready to solve
    """
    model = pyo.ConcreteModel(name="TSP_Master")

    # ── Decision variables — one binary per undirected edge ───────────────────
    model.x = pyo.Var(EDGES, domain=pyo.Binary)

    # Helper: cost of edge (i,j) stored with i < j
    def edge_cost(i, j):
        return D[i, j]  # symmetric, so D[i,j] == D[j,i]

    # ── Objective: minimize total tour distance ────────────────────────────────
    model.obj = pyo.Objective(
        expr=sum(edge_cost(i, j) * model.x[i, j] for (i, j) in EDGES),
        sense=pyo.minimize,
    )

    # ── Degree constraint — each node must have exactly 2 active edges ────────
    def edge_var(i, j):
        """Return the model variable for the undirected edge between i and j."""
        return model.x[min(i, j), max(i, j)]

    model.degree = pyo.ConstraintList()
    for i in range(N):
        model.degree.add(
            sum(edge_var(i, j) for j in range(N) if j != i) == 2
        )

    # ── Subtour elimination constraints (lazy) ────────────────────────────────
    model.sec = pyo.ConstraintList()
    for S in sec_cuts:
        model.sec.add(
            sum(model.x[i, j] for (i, j) in EDGES if i in S and j in S)
            <= len(S) - 1
        )

    return model

## 3. Subtour Detection

Given a binary solution $\{x^*_{ij}\}$, the degree constraints guarantee that every node has exactly **two** active edges. The solution graph is therefore a **disjoint union of undirected cycles**. We find these cycles by building an undirected adjacency list and tracing each connected component in order.

In [ ]:
def extract_solution(model: pyo.ConcreteModel, EDGES: list) -> dict:
    """Return a dict (i,j) → 0/1 from the solved Pyomo model (i < j for all keys)."""
    return {(i, j): round(pyo.value(model.x[i, j])) for (i, j) in EDGES}


def find_subtours(x_vals: dict, N: int) -> list[list[int]]:
    """
    Decompose an undirected binary solution into cycles (subtours).

    Each active edge (i,j) contributes to both i's and j's adjacency list.
    Since every node has degree exactly 2, the graph is a disjoint union of
    simple cycles. We trace each cycle by walking the adjacency list without
    retracing the edge we arrived on.

    Parameters
    ----------
    x_vals : dict (i,j) → 0/1, with i < j for all keys
    N      : number of nodes

    Returns
    -------
    subtours : list of lists; each sub-list is one cycle in traversal order
    """
    # Build undirected adjacency list
    adj: dict[int, list[int]] = {i: [] for i in range(N)}
    for (i, j), val in x_vals.items():
        if val > 0.5:
            adj[i].append(j)
            adj[j].append(i)

    visited: set[int] = set()
    subtours: list[list[int]] = []

    for start in range(N):
        if start in visited:
            continue
        # Trace the cycle: always move to the neighbor we didn't come from
        cycle = [start]
        visited.add(start)
        prev, cur = start, adj[start][0]   # pick an arbitrary first step
        while cur != start:
            visited.add(cur)
            cycle.append(cur)
            # next = the neighbor of cur that is not prev
            nxt = next(nb for nb in adj[cur] if nb != prev)
            prev, cur = cur, nxt
        subtours.append(cycle)

    return subtours


def tour_length(x_vals: dict, D: np.ndarray) -> float:
    """Compute the total distance of a solution (sum over active edges)."""
    return sum(D[i, j] * v for (i, j), v in x_vals.items() if v > 0.5)

## 4. Visualization Helpers

Two plotting functions:
- **`plot_iteration`** — draws the current solution, coloring each subtour a distinct color, and labels detected subtours.
- **`plot_final_tour`** — draws the single optimal tour.

In [ ]:
# Qualitative color palette for subtours
_PALETTE = list(mcolors.TABLEAU_COLORS.values())  # 10 distinct colors


def plot_iteration(
    coords: np.ndarray,
    x_vals: dict,
    subtours: list[list[int]],
    iteration: int,
    obj_val: float,
    n_cuts: int,
    ax: plt.Axes | None = None,
) -> None:
    """
    Plot one row-generation iteration (undirected edges).

    Each subtour is drawn in a distinct color as a closed polygon.
    If the solution is a single tour the title will say so.

    Parameters
    ----------
    ax : optional existing Axes to draw into (used for grid layout).
         If None, a new figure is created and shown immediately.
    """
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(5, 5))

    is_hamiltonian = len(subtours) == 1 and len(subtours[0]) == len(coords)

    for s_idx, subtour in enumerate(subtours):
        color = _PALETTE[s_idx % len(_PALETTE)]
        # Draw undirected edges as lines (closed cycle)
        cycle_coords = coords[subtour + [subtour[0]]]   # close the loop
        ax.plot(cycle_coords[:, 0], cycle_coords[:, 1],
                '-', color=color, lw=2.0, alpha=0.85, zorder=2)
        # Draw nodes
        ax.scatter(coords[subtour, 0], coords[subtour, 1],
                   s=160, color=color, zorder=3, edgecolors='white', linewidths=1.2)
        for node in subtour:
            ax.annotate(
                str(node),
                coords[node],
                textcoords='offset points',
                xytext=(6, 4),
                fontsize=9,
                fontweight='bold',
                color=color,
            )

    # Legend patch per subtour
    if not is_hamiltonian:
        patches = [
            mpatches.Patch(
                color=_PALETTE[s % len(_PALETTE)],
                label=f"Subtour {s+1}: (len={len(subtour)})",
            )
            for s, subtour in enumerate(subtours)
        ]
        ax.legend(handles=patches, fontsize=7, loc='upper left',
                  framealpha=0.9, title="Detected subtours")

    status = ("✓ Hamiltonian" if is_hamiltonian
              else f"{len(subtours)} subtours")
    ax.set_title(
        f"Iter {iteration}  |  Obj={obj_val:.2f}  |  SECs={n_cuts}\n{status}",
        fontsize=9,
    )
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.grid(True, alpha=0.25)

    if own_fig:
        plt.tight_layout()
        plt.show()


def plot_final_tour(
    coords: np.ndarray,
    x_vals: dict,
    obj_val: float,
    n_iterations: int,
    elapsed: float,
) -> None:
    """Draw the optimal undirected tour."""
    fig, ax = plt.subplots(figsize=(7, 6))
    color = 'royalblue'

    # Reconstruct tour order from undirected adjacency
    adj: dict[int, list[int]] = {i: [] for i in range(len(coords))}
    for (i, j), v in x_vals.items():
        if v > 0.5:
            adj[i].append(j)
            adj[j].append(i)

    order = [0]
    prev, cur = 0, adj[0][0]
    while cur != 0:
        order.append(cur)
        nxt = next(nb for nb in adj[cur] if nb != prev)
        prev, cur = cur, nxt
    order.append(0)   # close the loop

    cx = coords[order, 0]
    cy = coords[order, 1]
    ax.plot(cx, cy, '-', color=color, lw=2.0, alpha=0.7, zorder=1)

    ax.scatter(coords[:, 0], coords[:, 1], s=160, color=color, zorder=3,
               edgecolors='white', linewidths=1.2)
    for i, (x, y) in enumerate(coords):
        ax.annotate(str(i), (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=9, fontweight='bold')

    ax.set_title(
        f"Optimal TSP Tour  |  Total distance = {obj_val:.2f}\n"
        f"Solved in {n_iterations} iteration(s)  |  Wall time: {elapsed:.2f}s",
        fontsize=12,
    )
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 5. Row Generation Algorithm

We iterate until the undirected solution graph consists of a single cycle (all $N$ nodes in one connected component of degree 2).

In [ ]:
# ── Solver setup ──────────────────────────────────────────────────────────────
solver = pyo.SolverFactory('appsi_highs')
solver.options['mip_rel_gap'] = 1e-6
solver.options['log_to_console'] = False    # suppress HiGHS banner

# ── Row generation state ──────────────────────────────────────────────────────
sec_cuts:   list[frozenset] = []   # accumulated SEC cuts
history:    list[dict]      = []   # store per-iteration data for later analysis

MAX_ITER = 200
t_start  = time.perf_counter()

print(f"{'Iter':>5}  {'Obj':>12}  {'#Subtours':>10}  {'#SECs':>7}  {'Time(s)':>8}")
print("-" * 50)

for iteration in range(1, MAX_ITER + 1):

    # ── Solve master MIP ──────────────────────────────────────────────────────
    model = build_master_problem(N, D, EDGES, sec_cuts)
    result = solver.solve(model)

    if result.solver.termination_condition not in (
        pyo.TerminationCondition.optimal,
        pyo.TerminationCondition.feasible,
    ):
        print(f"Solver failed at iteration {iteration}: {result.solver.termination_condition}")
        break

    # ── Extract solution ──────────────────────────────────────────────────────
    x_vals   = extract_solution(model, EDGES)
    obj_val  = pyo.value(model.obj)
    subtours = find_subtours(x_vals, N)
    elapsed  = time.perf_counter() - t_start

    print(f"{iteration:>5}  {obj_val:>12.4f}  {len(subtours):>10}  {len(sec_cuts):>7}  {elapsed:>8.3f}")

    # ── Store history ─────────────────────────────────────────────────────────
    history.append({
        "iteration":  iteration,
        "obj":        obj_val,
        "n_subtours": len(subtours),
        "n_cuts":     len(sec_cuts),
        "x_vals":     x_vals.copy(),
        "subtours":   [list(s) for s in subtours],
    })

    # ── Termination check ─────────────────────────────────────────────────────
    if len(subtours) == 1:          # single tour → optimality
        print(f"\n✓ Optimal tour found after {iteration} iteration(s).")
        print(f"  Total distance : {obj_val:.4f}")
        print(f"  SECs added     : {len(sec_cuts)}")
        print(f"  Wall time      : {elapsed:.3f}s")
        break

    # ── Add SECs for every proper subtour detected ────────────────────────────
    new_cuts = 0
    for subtour in subtours:
        S = frozenset(subtour)
        if len(S) < N:              # skip if it already spans all nodes
            sec_cuts.append(S)
            new_cuts += 1
    if new_cuts == 0:
        print("No new cuts generated — terminating.")
        break

else:
    print(f"Warning: reached maximum iterations ({MAX_ITER}) without finding a tour.")

In [ ]:

# ── Plot all iterations in a 3-column grid ────────────────────────────────────
import math

N_COLS = 3
n_iters = len(history)
n_rows  = math.ceil(n_iters / N_COLS)

fig, axes = plt.subplots(
    n_rows, N_COLS,
    figsize=(6 * N_COLS, 5 * n_rows),
    squeeze=False,
)

for idx, h in enumerate(history):
    row, col = divmod(idx, N_COLS)
    ax = axes[row][col]
    plot_iteration(
        coords    = coords,
        x_vals    = h["x_vals"],
        subtours  = h["subtours"],
        iteration = h["iteration"],
        obj_val   = h["obj"],
        n_cuts    = h["n_cuts"],
        ax        = ax,
    )

# Hide any unused axes in the last row
for empty_col in range(n_iters % N_COLS or N_COLS, N_COLS):
    axes[n_rows - 1][empty_col].set_visible(False)

plt.suptitle("Row Generation — All Iterations", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 6. Final Optimal Tour

In [ ]:
# ── Plot the final optimal tour ───────────────────────────────────────────────
final = history[-1]
plot_final_tour(
    coords      = coords,
    x_vals      = final["x_vals"],
    obj_val     = final["obj"],
    n_iterations= final["iteration"],
    elapsed     = elapsed,
)

## 7. Convergence Analysis

We visualize three things across iterations:
1. **Objective value** 
2. **Number of subtours**
3. **Cumulative SECs added**

In [ ]:
iters      = [h["iteration"]   for h in history]
objs       = [h["obj"]         for h in history]
n_subtours = [h["n_subtours"]  for h in history]
n_cuts     = [h["n_cuts"] + (h["n_subtours"] if h["n_subtours"] > 1 else 0)
              for h in history]   # total cuts after this iteration

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Objective ─────────────────────────────────────────────────────────────────
axes[0].plot(iters, objs, 'o-', color='steelblue', lw=2, ms=6)
axes[0].axhline(objs[-1], color='salmon', ls='--', lw=1.5, label=f'Optimal = {objs[-1]:.2f}')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Objective value')
axes[0].set_title('Objective vs Iteration')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# ── Number of subtours ────────────────────────────────────────────────────────
axes[1].bar(iters, n_subtours, color='coral', edgecolor='white', width=0.6)
axes[1].axhline(1, color='steelblue', ls='--', lw=1.5, label='Target: 1 tour')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Number of subtours')
axes[1].set_title('Subtours Detected vs Iteration')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3, axis='y')

# ── Cumulative SECs ───────────────────────────────────────────────────────────
axes[2].step(iters, n_cuts, where='post', color='mediumpurple', lw=2)
axes[2].fill_between(iters, n_cuts, step='post', alpha=0.25, color='mediumpurple')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Cumulative SECs')
axes[2].set_title('SECs Added vs Iteration')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Row Generation Convergence', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()